In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, vonmises_fisher
import seaborn as sns
import sys
import glob
from tqdm import trange 

In [3]:
sys.path.append("../src")
from sphere import *

In [4]:
sys.path.append("../../bosporus-package/")
from bosporus import *

In [5]:
edge_type = "rnn"
n = 5000
coords = sample_von_mises_fisher(n=n, kappas=[1, 3, 5])

In [6]:
k = None
cap_radius = 1
radius = 0.1

In [9]:
N = len(coords)
global_edges = get_edge_list(coords, edge_type, k=k, r=radius)

In [11]:
len(global_edges) / N

9.103841536614645

In [ ]:
wewqewqe

In [ ]:
global_centralities = pd.DataFrame(
    compute_centrality_measures(global_edges, N)
)
measures = global_centralities.columns

In [ ]:
crop, distances = crop_cap(coords, cap_radius)
crop_coords = coords[crop]

In [ ]:
distances

In [ ]:
local_edges = get_edge_list(crop_coords, edge_type, k=k, r=r)        
BOSPORUS_results = get_bosporus_corrections(crop_coords, local_edges, measures, distances)
sern_median = get_sern_median(crop_coords, local_edges)

In [ ]:
np.sum(sern_median["closeness"].isna())

In [ ]:
results = pd.concat(
    {
        "original": global_centralities.loc[crop].sort_index(axis=0).sort_index(axis=1).reset_index(drop=True),
        "crop": BOSPORUS_results[measures].sort_index(axis=0).sort_index(axis=1),
        "distance": BOSPORUS_results["distance_to_cap"],
        "BOSPORUS_corrections": BOSPORUS_results.filter(like="BOSPORUS", axis=1).sort_index(axis=0).sort_index(axis=1),
        "sern": sern_median.sort_index(axis=0).sort_index(axis=1),
        "sern_corrected": (
            BOSPORUS_results[measures].sort_index(axis=0).sort_index(axis=1)
            - sern_median.sort_index(axis=0).sort_index(axis=1)
        ),
    },
    axis=1,
)

In [ ]:
# --- correlations ---
corrs_original_crop, corrs_original_corrected = [], []
corrs_original_sern, corrs_crop_sern = [], []

for m in measures:
    corrs_original_crop.append(
        pearsonr(results["original"][m], results["crop"][m]).statistic
    )
    corrs_original_corrected.append(
        pearsonr(results["original"][m], results["BOSPORUS_corrections"][f"BOSPORUS corrected {m}"]).statistic
    )
    corrs_original_sern.append(
        pearsonr(results["original"][m], results["sern_corrected"][m]).statistic
    )
    corrs_crop_sern.append(
        pearsonr(results["crop"][m], results["sern"][m]).statistic
    )

correlations = pd.DataFrame(index=measures)
correlations["original vs. on crop"] = corrs_original_crop
correlations["original vs. BOSPORUS corrected on crop"] = corrs_original_corrected
correlations["original vs. SERN corrected on crop"] = corrs_original_sern
correlations["on crop vs. SERN values"] = corrs_crop_sern
correlations["cap_radius"] = cap_radius

In [ ]:
correlations

In [ ]:
plt.scatter(BOSPORUS_results["distance_to_cap"], BOSPORUS_results["degree"])

In [ ]:
plt.scatter(BOSPORUS_results["distance_to_cap"], BOSPORUS_results["BOSPORUS corrected degree"])

In [ ]:
plt.scatter(BOSPORUS_results["distance_to_cap"], global_centralities.loc[crop].sort_index(axis=0).sort_index(axis=1).reset_index(drop=True)["degree"])